# Final Task B 17-class - EfficientNet-B3

Train EfficientNet-B3 directly on the final 17 species classes. This uses ImageNet pretrained initialization and does not load the old 20-class Task B checkpoints.


In [1]:
from pathlib import Path
import sys

if Path('/content').exists():
    ROOT = Path('/content/woodbench')
else:
    ROOT = next((parent for parent in [Path.cwd(), *Path.cwd().parents] if (parent / 'src').exists()), Path.cwd())

sys.path.insert(0, str(ROOT))

import os
os.chdir(ROOT)
FINAL_B_DIR = ROOT / "notebooks" / "phase 1" / "task-B-17"
sys.path.insert(0, str(FINAL_B_DIR))
print(f"ROOT: {ROOT}")
print(f"task-B-17: {FINAL_B_DIR}")


ROOT: /workspace/woodbench
final-B: /workspace/woodbench/notebooks/phase 1/final-B


In [4]:
# Colab-only setup. Cloud data is expected to be already converted to 17 classes
# by 00_data_preparation.ipynb. This download only runs when the dataset is absent.
if Path('/content').exists():
    %pip install -q -r {ROOT}/requirements.txt

    DATA_DIR = ROOT / "data" / "task-B-group-split"
    if not DATA_DIR.exists():
        from huggingface_hub import login, snapshot_download
        import os

        token = os.getenv("HF_TOKEN", "")
        if token:
            login(token=token)
        snapshot_download(
            repo_id="ny0g507h3p/wood-task-B",
            repo_type="dataset",
            local_dir=str(DATA_DIR),
            ignore_patterns=["*.md", "README*"],
        )


In [5]:
from final_b_17class import (
    BACKBONE_CONFIGS,
    prepare_environment,
    run_all_experiments,
    summarize_results,
    verify_17class_dataset,
)

prepare_environment(ROOT)

BACKBONE = "efficientnet_b3"
DISPLAY_NAME = "EfficientNet-B3"
DATA_ROOT = ROOT / "data" / "task-B-group-split"
BATCH_SIZE = BACKBONE_CONFIGS[BACKBONE]["batch_size"]
EPOCHS = 50

print(f"Backbone: {DISPLAY_NAME} ({BACKBONE})")
print(f"Batch size: {BATCH_SIZE}")
print(f"Data root: {DATA_ROOT}")


Backbone: EfficientNet-B3 (efficientnet_b3)
Batch size: 32
Data root: /workspace/woodbench/data/task-B-group-split


In [6]:
class_names = verify_17class_dataset(DATA_ROOT)
print(f"Classes ({len(class_names)}):")
for idx, class_name in enumerate(class_names):
    print(f"  {idx:02d}: {class_name}")


[Data] train: 17 classes, 3365 images
[Data] val: 17 classes, 903 images
[Data] test: 17 classes, 532 images
Classes (17):
  00: Araucaria araucana
  01: Callitris columellaris
  02: Calocedrus rupestris
  03: Chamaecyparis obtusa
  04: Cryptomeria japonica
  05: Cunninghamia lanceolata
  06: Fokienia hodginsii
  07: Picea glauca
  08: Pinus cernua
  09: Pinus dalatensis
  10: Pinus krempfii
  11: Pinus radiata
  12: Podocarpus pilgeri
  13: Pseudotsuga brevifolia
  14: Taxus wallichiana
  15: Thujopsis dolabrata
  16: Xanthocyparis vietnamensis


In [ ]:
import torch

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
if DEVICE == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")
else:
    print("CUDA not available; CPU training will be slow.")

results = run_all_experiments(
    backbone_name=BACKBONE,
    root=ROOT,
    data_root=DATA_ROOT,
    batch_size=BATCH_SIZE,
    epochs=EPOCHS,
    device=DEVICE,
)


GPU: NVIDIA GeForce RTX 4090
VRAM: 23.5 GB
[Data] train: 17 classes, 3365 images
[Data] val: 17 classes, 903 images
[Data] test: 17 classes, 532 images
[Device] cuda
[Output] /workspace/woodbench/data/experiments/Task-B-17class

main/seed_42/efficientnet_b3
[main | efficientnet_b3 s42] train=3365, val=903, test=532, classes=17


model.safetensors:   0%|          | 0.00/49.3M [00:00<?, ?B/s]

Epoch 1/50 | Train Loss: 2.8302, Acc: 7.26% | Val Loss: 2.8321, Acc: 5.20% | LR: 2.00e-05[Best][EarlyStop 0/10]
  ↳ New best! Saved to efficientnet_b3_seed42_best.pth (Val Acc: 5.20%)


Epoch 2/50 | Train Loss: 2.7569, Acc: 20.42% | Val Loss: 2.7099, Acc: 29.24% | LR: 4.00e-05[Best][EarlyStop 0/10]
  ↳ New best! Saved to efficientnet_b3_seed42_best.pth (Val Acc: 29.24%)


Epoch 3/50 | Train Loss: 2.5292, Acc: 49.97% | Val Loss: 2.4597, Acc: 59.36% | LR: 6.00e-05[Best][EarlyStop 0/10]
  ↳ New best! Saved to efficientnet_b3_seed42_best.pth (Val Acc: 59.36%)


Epoch 4/50 | Train Loss: 2.0710, Acc: 67.89% | Val Loss: 1.9957, Acc: 67.66% | LR: 8.00e-05[Best][EarlyStop 0/10]
  ↳ New best! Saved to efficientnet_b3_seed42_best.pth (Val Acc: 67.66%)


... (88 outputs left out)

Epoch 3/50 | Train Loss: 2.5287, Acc: 52.92% | Val Loss: 2.4790, Acc: 57.14% | LR: 6.00e-05[Best][EarlyStop 0/10]
  ↳ New best! Saved to efficientnet_b3_seed456_best.pth (Val Acc: 57.14%)


Epoch 4/50 | Train Loss: 2.0766, Acc: 71.19% | Val Loss: 2.0143, Acc: 66.89% | LR: 8.00e-05[Best][EarlyStop 0/10]
  ↳ New best! Saved to efficientnet_b3_seed456_best.pth (Val Acc: 66.89%)


Epoch 5/50 | Train Loss: 1.4926, Acc: 79.38% | Val Loss: 1.4359, Acc: 77.74% | LR: 1.00e-04[Best][EarlyStop 0/10]
  ↳ New best! Saved to efficientnet_b3_seed456_best.pth (Val Acc: 77.74%)


Epoch 6/50 [Train]:   0%|          | 0/105 [00:00<?, ?batch/s]

In [ ]:
summarize_results(results)


## Continue after Vast interruption

Run the root, import/config, and data-check cells above, but do not rerun the old `run_all_experiments` cell. Then run each config cell below independently. Completed metrics are reused; interrupted checkpoints restore model, optimizer, and scheduler state.

In [7]:
import torch

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
if DEVICE == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")
else:
    print("CUDA not available; CPU training will be slow.")


GPU: NVIDIA GeForce RTX 4090
VRAM: 23.5 GB


In [8]:
from final_b_17class import EXPERIMENTS, load_completed_results, train_one

continued_results = {
    (row["experiment"], row["seed"]): row
    for row in load_completed_results(backbone_name=BACKBONE, root=ROOT)
}

def run_remaining(experiment_name, seed):
    result = train_one(
        root=ROOT,
        backbone=BACKBONE,
        seed=seed,
        experiment_name=experiment_name,
        experiment_cfg=EXPERIMENTS[experiment_name],
        data_root=DATA_ROOT,
        batch_size=BATCH_SIZE,
        epochs=EPOCHS,
        device=DEVICE,
        resume_training=True,
    )
    continued_results[(experiment_name, seed)] = result
    return result

print(f"Loaded {len(continued_results)}/6 completed configs for {BACKBONE}.")


Loaded 2/6 completed configs for efficientnet_b3.


In [9]:
# Resume main seed 456 from its saved checkpoint
run_remaining("main", 456)


[main | efficientnet_b3 s456] train=3365, val=903, test=532, classes=17
[Resume] /workspace/woodbench/data/experiments/Task-B-17class/main/seed_456/efficientnet_b3/efficientnet_b3_seed456_best.pth -> epoch 7/50


Epoch 7/50 | Train Loss: 0.9076, Acc: 90.65% | Val Loss: 1.0158, Acc: 85.38% | LR: 9.95e-05[EarlyStop 1/10]


Epoch 8/50 | Train Loss: 0.8298, Acc: 93.21% | Val Loss: 0.9175, Acc: 90.03% | LR: 9.89e-05[Best][EarlyStop 0/10]
  ↳ New best! Saved to efficientnet_b3_seed456_best.pth (Val Acc: 90.03%)


Epoch 9/50 | Train Loss: 0.7860, Acc: 94.46% | Val Loss: 0.8692, Acc: 90.92% | LR: 9.81e-05[Best][EarlyStop 0/10]
  ↳ New best! Saved to efficientnet_b3_seed456_best.pth (Val Acc: 90.92%)


Epoch 10/50 | Train Loss: 0.7661, Acc: 95.42% | Val Loss: 0.8491, Acc: 91.47% | LR: 9.70e-05[Best][EarlyStop 0/10]
  ↳ New best! Saved to efficientnet_b3_seed456_best.pth (Val Acc: 91.47%)


Epoch 11/50 | Train Loss: 0.7326, Acc: 96.52% | Val Loss: 0.8198, Acc: 91.36% | LR: 9.57e-05[EarlyStop 1/10]


... (23 outputs left out)

Epoch 35/50 | Train Loss: 0.6395, Acc: 98.96% | Val Loss: 0.7030, Acc: 95.46% | LR: 2.50e-05[EarlyStop 8/10]


Epoch 36/50 | Train Loss: 0.6341, Acc: 99.29% | Val Loss: 0.7056, Acc: 95.90% | LR: 2.20e-05[EarlyStop 9/10]


Epoch 37/50 | Train Loss: 0.6352, Acc: 99.02% | Val Loss: 0.6938, Acc: 96.12% | LR: 1.92e-05[EarlyStop 10/10]

[EarlyStopping] No improvement for 10 epochs. Stopping at epoch 37.
[main | efficientnet_b3 s456] val_acc=96.23% test_acc=97.37% F1=97.42%


{'backbone': 'efficientnet_b3',
 'seed': 456,
 'experiment': 'main',
 'num_classes': 17,
 'class_names': ['Araucaria araucana',
  'Callitris columellaris',
  'Calocedrus rupestris',
  'Chamaecyparis obtusa',
  'Cryptomeria japonica',
  'Cunninghamia lanceolata',
  'Fokienia hodginsii',
  'Picea glauca',
  'Pinus cernua',
  'Pinus dalatensis',
  'Pinus krempfii',
  'Pinus radiata',
  'Podocarpus pilgeri',
  'Pseudotsuga brevifolia',
  'Taxus wallichiana',
  'Thujopsis dolabrata',
  'Xanthocyparis vietnamensis'],
 'best_val_acc': 96.23477297895903,
 'best_epoch': 27,
 'final_train_acc': 99.01785714285714,
 'final_val_acc': 96.12403100775194,
 'stopped_early': True,
 'epochs_trained': 37,
 'test_accuracy': 0.9736842105263158,
 'test_f1_macro': 0.9741682908313551,
 'test_precision_macro': 0.9762933996454957,
 'test_recall_macro': 0.9740464454404348,
 'test_f1_per_class': {'Araucaria araucana': 1.0,
  'Callitris columellaris': 0.9850746268656716,
  'Calocedrus rupestris': 1.0,
  'Chamaecypa

In [10]:
# Remaining 90-degree rotation ablation
run_remaining("taskB_90rotation", 42)


[taskB_90rotation | efficientnet_b3 s42] train=3365, val=903, test=532, classes=17


Epoch 1/50 | Train Loss: 2.8372, Acc: 7.29% | Val Loss: 2.8251, Acc: 8.42% | LR: 2.00e-05[Best][EarlyStop 0/10]
  ↳ New best! Saved to efficientnet_b3_seed42_best.pth (Val Acc: 8.42%)


Epoch 2/50 | Train Loss: 2.7484, Acc: 23.04% | Val Loss: 2.6319, Acc: 46.18% | LR: 4.00e-05[Best][EarlyStop 0/10]
  ↳ New best! Saved to efficientnet_b3_seed42_best.pth (Val Acc: 46.18%)


Epoch 3/50 | Train Loss: 2.4946, Acc: 56.25% | Val Loss: 2.2309, Acc: 72.31% | LR: 6.00e-05[Best][EarlyStop 0/10]
  ↳ New best! Saved to efficientnet_b3_seed42_best.pth (Val Acc: 72.31%)


Epoch 4/50 | Train Loss: 2.0018, Acc: 73.18% | Val Loss: 1.5911, Acc: 81.17% | LR: 8.00e-05[Best][EarlyStop 0/10]
  ↳ New best! Saved to efficientnet_b3_seed42_best.pth (Val Acc: 81.17%)


Epoch 5/50 | Train Loss: 1.4278, Acc: 80.15% | Val Loss: 1.1081, Acc: 87.49% | LR: 1.00e-04[Best][EarlyStop 0/10]
  ↳ New best! Saved to efficientnet_b3_seed42_best.pth (Val Acc: 87.49%)


... (39 outputs left out)

Epoch 45/50 | Train Loss: 0.6301, Acc: 99.38% | Val Loss: 0.6567, Acc: 97.67% | LR: 3.02e-06[EarlyStop 9/10]


Epoch 46/50 | Train Loss: 0.6335, Acc: 99.20% | Val Loss: 0.6621, Acc: 96.90% | LR: 1.94e-06[EarlyStop 10/10]

[EarlyStopping] No improvement for 10 epochs. Stopping at epoch 46.


[taskB_90rotation | efficientnet_b3 s42] val_acc=97.67% test_acc=98.87% F1=98.93%


{'backbone': 'efficientnet_b3',
 'seed': 42,
 'experiment': 'taskB_90rotation',
 'num_classes': 17,
 'class_names': ['Araucaria araucana',
  'Callitris columellaris',
  'Calocedrus rupestris',
  'Chamaecyparis obtusa',
  'Cryptomeria japonica',
  'Cunninghamia lanceolata',
  'Fokienia hodginsii',
  'Picea glauca',
  'Pinus cernua',
  'Pinus dalatensis',
  'Pinus krempfii',
  'Pinus radiata',
  'Podocarpus pilgeri',
  'Pseudotsuga brevifolia',
  'Taxus wallichiana',
  'Thujopsis dolabrata',
  'Xanthocyparis vietnamensis'],
 'best_val_acc': 97.67441860465117,
 'best_epoch': 36,
 'final_train_acc': 99.19642857142857,
 'final_val_acc': 96.89922480620154,
 'stopped_early': True,
 'epochs_trained': 46,
 'test_accuracy': 0.9887218045112782,
 'test_f1_macro': 0.9892892298641497,
 'test_precision_macro': 0.9902683432095197,
 'test_recall_macro': 0.9895676219205631,
 'test_f1_per_class': {'Araucaria araucana': 1.0,
  'Callitris columellaris': 1.0,
  'Calocedrus rupestris': 1.0,
  'Chamaecyparis 

In [11]:
# Remaining label-smoothing=0.0 ablation
run_remaining("taskB_label_smoothing_0.0", 42)


[taskB_label_smoothing_0.0 | efficientnet_b3 s42] train=3365, val=903, test=532, classes=17


Epoch 1/50 | Train Loss: 2.8289, Acc: 7.26% | Val Loss: 2.8309, Acc: 5.20% | LR: 2.00e-05[Best][EarlyStop 0/10]
  ↳ New best! Saved to efficientnet_b3_seed42_best.pth (Val Acc: 5.20%)


Epoch 2/50 | Train Loss: 2.7473, Acc: 20.48% | Val Loss: 2.6946, Acc: 29.24% | LR: 4.00e-05[Best][EarlyStop 0/10]
  ↳ New best! Saved to efficientnet_b3_seed42_best.pth (Val Acc: 29.24%)


Epoch 3/50 | Train Loss: 2.4910, Acc: 50.00% | Val Loss: 2.4112, Acc: 59.14% | LR: 6.00e-05[Best][EarlyStop 0/10]
  ↳ New best! Saved to efficientnet_b3_seed42_best.pth (Val Acc: 59.14%)


Epoch 4/50 | Train Loss: 1.9568, Acc: 67.65% | Val Loss: 1.8640, Acc: 67.44% | LR: 8.00e-05[Best][EarlyStop 0/10]
  ↳ New best! Saved to efficientnet_b3_seed42_best.pth (Val Acc: 67.44%)


Epoch 5/50 | Train Loss: 1.2452, Acc: 78.01% | Val Loss: 1.1486, Acc: 76.74% | LR: 1.00e-04[Best][EarlyStop 0/10]
  ↳ New best! Saved to efficientnet_b3_seed42_best.pth (Val Acc: 76.74%)


... (43 outputs left out)

Epoch 49/50 | Train Loss: 0.0466, Acc: 98.48% | Val Loss: 0.1125, Acc: 95.90% | LR: 1.22e-07[EarlyStop 5/10]


Epoch 50/50 | Train Loss: 0.0388, Acc: 98.81% | Val Loss: 0.1056, Acc: 96.46% | LR: 0.00e+00[EarlyStop 6/10]


[taskB_label_smoothing_0.0 | efficientnet_b3 s42] val_acc=96.68% test_acc=97.18% F1=97.18%


{'backbone': 'efficientnet_b3',
 'seed': 42,
 'experiment': 'taskB_label_smoothing_0.0',
 'num_classes': 17,
 'class_names': ['Araucaria araucana',
  'Callitris columellaris',
  'Calocedrus rupestris',
  'Chamaecyparis obtusa',
  'Cryptomeria japonica',
  'Cunninghamia lanceolata',
  'Fokienia hodginsii',
  'Picea glauca',
  'Pinus cernua',
  'Pinus dalatensis',
  'Pinus krempfii',
  'Pinus radiata',
  'Podocarpus pilgeri',
  'Pseudotsuga brevifolia',
  'Taxus wallichiana',
  'Thujopsis dolabrata',
  'Xanthocyparis vietnamensis'],
 'best_val_acc': 96.67774086378738,
 'best_epoch': 44,
 'final_train_acc': 98.80952380952381,
 'final_val_acc': 96.4562569213732,
 'stopped_early': False,
 'epochs_trained': 50,
 'test_accuracy': 0.9718045112781954,
 'test_f1_macro': 0.9717945441250974,
 'test_precision_macro': 0.9729950745263759,
 'test_recall_macro': 0.9720159763900694,
 'test_f1_per_class': {'Araucaria araucana': 1.0,
  'Callitris columellaris': 0.9538461538461539,
  'Calocedrus rupestris'

In [12]:
# Remaining label-smoothing=0.2 ablation
run_remaining("taskB_label_smoothing_0.2", 42)


[taskB_label_smoothing_0.2 | efficientnet_b3 s42] train=3365, val=903, test=532, classes=17


Epoch 1/50 | Train Loss: 2.8314, Acc: 7.26% | Val Loss: 2.8332, Acc: 5.20% | LR: 2.00e-05[Best][EarlyStop 0/10]
  ↳ New best! Saved to efficientnet_b3_seed42_best.pth (Val Acc: 5.20%)


Epoch 2/50 | Train Loss: 2.7665, Acc: 20.51% | Val Loss: 2.7251, Acc: 29.35% | LR: 4.00e-05[Best][EarlyStop 0/10]
  ↳ New best! Saved to efficientnet_b3_seed42_best.pth (Val Acc: 29.35%)


Epoch 3/50 | Train Loss: 2.5673, Acc: 50.09% | Val Loss: 2.5079, Acc: 59.25% | LR: 6.00e-05[Best][EarlyStop 0/10]
  ↳ New best! Saved to efficientnet_b3_seed42_best.pth (Val Acc: 59.25%)


Epoch 4/50 | Train Loss: 2.1825, Acc: 68.24% | Val Loss: 2.1237, Acc: 67.88% | LR: 8.00e-05[Best][EarlyStop 0/10]
  ↳ New best! Saved to efficientnet_b3_seed42_best.pth (Val Acc: 67.88%)


Epoch 5/50 | Train Loss: 1.7325, Acc: 78.96% | Val Loss: 1.6936, Acc: 79.07% | LR: 1.00e-04[Best][EarlyStop 0/10]
  ↳ New best! Saved to efficientnet_b3_seed42_best.pth (Val Acc: 79.07%)


... (28 outputs left out)

Epoch 34/50 | Train Loss: 1.0742, Acc: 99.08% | Val Loss: 1.1313, Acc: 95.13% | LR: 2.81e-05[EarlyStop 9/10]


Epoch 35/50 | Train Loss: 1.0734, Acc: 98.75% | Val Loss: 1.1235, Acc: 95.46% | LR: 2.50e-05[EarlyStop 10/10]

[EarlyStopping] No improvement for 10 epochs. Stopping at epoch 35.


[taskB_label_smoothing_0.2 | efficientnet_b3 s42] val_acc=95.68% test_acc=96.05% F1=96.06%


{'backbone': 'efficientnet_b3',
 'seed': 42,
 'experiment': 'taskB_label_smoothing_0.2',
 'num_classes': 17,
 'class_names': ['Araucaria araucana',
  'Callitris columellaris',
  'Calocedrus rupestris',
  'Chamaecyparis obtusa',
  'Cryptomeria japonica',
  'Cunninghamia lanceolata',
  'Fokienia hodginsii',
  'Picea glauca',
  'Pinus cernua',
  'Pinus dalatensis',
  'Pinus krempfii',
  'Pinus radiata',
  'Podocarpus pilgeri',
  'Pseudotsuga brevifolia',
  'Taxus wallichiana',
  'Thujopsis dolabrata',
  'Xanthocyparis vietnamensis'],
 'best_val_acc': 95.68106312292359,
 'best_epoch': 25,
 'final_train_acc': 98.75,
 'final_val_acc': 95.45957918050941,
 'stopped_early': True,
 'epochs_trained': 35,
 'test_accuracy': 0.9605263157894737,
 'test_f1_macro': 0.9605870257855684,
 'test_precision_macro': 0.9619346337489885,
 'test_recall_macro': 0.961462780066977,
 'test_f1_per_class': {'Araucaria araucana': 1.0,
  'Callitris columellaris': 0.9206349206349206,
  'Calocedrus rupestris': 1.0,
  'Cha

In [13]:
results = list(continued_results.values())
summarize_results(results)



Task B 17-class summary
experiment                       seed   val_acc  test_acc        f1      prec       rec
----------------------------------------------------------------------------------------
main                               42    95.79%    97.37%    97.42%    97.51%    97.44%
main                              123    96.57%    97.37%    97.42%    97.63%    97.40%
main                              456    96.23%    97.37%    97.42%    97.63%    97.40%
taskB_90rotation                   42    97.67%    98.87%    98.93%    99.03%    98.96%
taskB_label_smoothing_0.0          42    96.68%    97.18%    97.18%    97.30%    97.20%
taskB_label_smoothing_0.2          42    95.68%    96.05%    96.06%    96.19%    96.15%
----------------------------------------------------------------------------------------
main test_acc: 97.37 +/- 0.00%
main test_f1:  97.42 +/- 0.00%
